<a href="https://colab.research.google.com/github/Haniiko1/mol_visualize/blob/main/conformational_analysis_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img align="right" src="https://github.com/codeBMB/April26_Rutgers-RCSB/raw/main/images/PyBMB_logo.png" width="150" height="150" />

# Project: Organic Chemistry Computational Tools
## Notebook: Molecular Conformation & Rotational Energy Profiles

**Purpose:**
This notebook introduces students to molecular conformation — the different three-dimensional
shapes a molecule can adopt by rotating around single bonds. Using computational chemistry
tools, students will build 3D molecular models, visualize conformational changes interactively,
and generate rotational energy profiles. By the end, students will be able to connect familiar
concepts from lecture (Newman projections, staggered vs. eclipsed conformations, torsional
strain) to quantitative energy data.

**Input Data:**
* **Molecules:** Defined internally via SMILES strings (no external files required).
* **Force Field:** MMFF94 (Merck Molecular Force Field), accessed via RDKit.
* **Retrieved On:** N/A — all molecules generated computationally.

**Libraries:**
* `rdkit` — Cheminformatics toolkit for parsing SMILES, generating 3D coordinates, and computing
  molecular mechanics energies.
* `py3Dmol` — Browser-based 3D molecular visualization.
* `ipywidgets` — Interactive slider and layout widgets for Colab notebooks.
* `matplotlib` — Plotting the conformational energy profile.

**Status with Date:** Complete — 2025-06

**License:**

<img src="https://github.com/codeBMB/April26_Rutgers-RCSB/raw/main/images/by-nc-sa.png" width="100"/>
CC BY-NC-SA — reusers may distribute, remix, adapt, and build upon
the material for noncommercial purposes only, with attribution,
under identical terms.

---
**Authorship:** Hieu Nguyen, Wally Novak

**Acknowledgements:**

**Contact Info:**

# 0. Explanation of Colab and how to run (in notebooks for the first workshop)

This is an interactive notebook. Each **code cell** contains Python code that you run by
clicking the ▶ play button on the left side of the cell, or by pressing **Shift+Enter**.

![run button image](https://github.com/wallynovak/biochemistry_seq_analysis/blob/main/images/run.png?raw=1)

- A spinning circle ◌ means the cell is still running — wait for it to finish.
- A checkmark ✓ and a number like `[1]` in brackets means the cell completed successfully.
- If you see a red error message, re-read it carefully — it usually tells you exactly what went wrong.

---

In [ ]:
#@title Cell 0: Install Packages
# ipympl enables interactive matplotlib figures inside Colab widgets.
# rdkit and py3Dmol are the core cheminformatics and visualization tools.

import subprocess, sys

def install_if_missing(package, import_name=None):
    import_name = import_name or package
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

install_if_missing("ipympl")
install_if_missing("rdkit", "rdkit")
install_if_missing("py3Dmol")
get_ipython().kernel.do_shutdown(restart=True)

# After ipympl installs for the first time, a kernel restart may be required.

# If there is a pop up message saying your session crashed for an unknown reason, don't worry because it basically means that the session restarted to finish installing packages. You can continue to run the next code cell below.

In [ ]:
#@title Cell 0.1: Import Libraries

import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from rdkit import Chem
from rdkit.Chem import Draw, AllChem, rdMolTransforms
import py3Dmol

try:
    from google.colab import output, drive
    output.enable_custom_widget_manager()
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("All libraries imported successfully.")

# 1. Learning Goals

By the end of this notebook, you will be able to:

1. **Build a 3D molecular model** computationally and visualize it interactively.
2. **Define a dihedral angle** and explain how it describes molecular conformation.
3. **Connect energy profile features** (minima, maxima, barrier height) to physical concepts
   like staggered, eclipsed, and gauche conformations.
4. **Predict relative stability** of conformations from a rotational energy profile.
5. **Apply conformational analysis** to molecules beyond ethane and butane.

**Relevant Course Topics:** Conformational isomers, Newman projections, torsional strain,
steric strain, rotational barriers, VSEPR geometry.

---

# 2. Background: Molecular Conformation

## 2a. What is Conformation?

Molecules are not rigid structures. Atoms connected by **single bonds** can rotate relative
to one another, producing different spatial arrangements called **conformations**. Unlike
constitutional isomers (different connectivity) or stereoisomers (different fixed arrangements
in space), conformers are interconvertible at room temperature through bond rotation.

The energy cost of a full 360° rotation around a bond is captured in a **rotational energy
profile** (also called a torsional potential energy surface). Key features:

| Feature | Meaning |
|---|---|
| **Energy minimum** | A stable conformation (molecule "prefers" to be here) |
| **Energy maximum** | An unstable transition state between conformations |
| **Barrier height (ΔE)** | Energy required to rotate through the highest-energy point |




## 2b. Newman Projections

A **Newman projection** is a 2D representation of a molecule viewed directly along a bond, usually a carbon–carbon single bond. It is a useful tool for visualizing conformations and understanding how bond rotation affects molecular stability.

In a Newman projection:

- The **front atom** is represented by a dot (●).
- The **back atom** is represented by a circle (○).
- Bonds attached to each atom are drawn at 120° angles around the dot or circle.
- The angle between substituents on adjacent atoms corresponds to the **dihedral angle** (Section 2c).

Because the molecule is viewed directly down the rotating bond, Newman projections provide a simple way to distinguish between **staggered**, **gauche**, **anti**, and **eclipsed** conformations.

For butane:

| Conformation | Dihedral Angle | Relative Stability |
|---|---|---|
| Anti | 180° | Most stable (lowest energy) |
| Gauche | 60° or 300° | Stable local minimum |
| Eclipsed | 120° or 240° | Higher energy |
| Fully Eclipsed | 0° | Least stable (highest energy) |

Newman projections provide a direct visual interpretation of the rotational energy profile. As the dihedral angle changes during bond rotation, the molecule moves between energy minima (stable conformations) and energy maxima (unstable conformations).

<img src="https://cdn1.byjus.com/wp-content/uploads/2022/12/Conformers-01.png" width="400"> <img src="https://github.com/Haniiko1/mol_visualize/blob/main/ethane_dihedral_60.png?raw=true" width="200">

Source: https://byjus.com/chemistry/conformers/




## 2c. What is a Dihedral Angle?


A dihedral angle is defined by **four atoms in sequence** (A–B–C–D):
- Look along the B–C bond (the "central" rotating bond).
- Measure the angle between the A–B plane and the C–D plane.
- This is exactly what a **Newman projection** draws!

A dihedral of 0° = eclipsed; 60° = staggered (gauche); 180° = anti (for butane, A and D are two methyl groups).
<img src="https://cdn.masterorganicchemistry.com/wp-content/uploads/2020/02/11-Definition-of-the-dihedral-angle.gif" width="400">

Source: https://www.masterorganicchemistry.com/2020/02/28/staggered-vs-eclipsed-conformations-of-ethane/



In [ ]:
#@title Cell 2.1:Example of a 60° Dihedral Angle (3D Visualization)
# ── 1. Build & optimise ethane ───────────────────────────────────────────────
mol = Chem.MolFromSmiles("CC")
mol = Chem.AddHs(mol)
params = AllChem.ETKDGv3() # Create the EmbedParameters object
params.randomSeed = 42 # Set the randomSeed on the parameters object
AllChem.EmbedMolecule(mol, params) # Pass the parameters object to EmbedMolecule
AllChem.MMFFOptimizeMolecule(mol)
conf = mol.GetConformer()


def pos(i):
    p = conf.GetAtomPosition(i)
    return np.array([p.x, p.y, p.z])


c0, c1 = pos(0), pos(1)
h_c0 = [pos(i) for i in (2, 3, 4)]
h_c1 = [pos(i) for i in (5, 6, 7)]


# ── 2. Draw a precise chemical plane ─────────────────────────────────────────
def draw_dihedral_plane(view, c_center, c_attached, h_attached, color, size=1.2):
    """Draws a grid plane neatly aligned with the C-C bond and a C-H bond.

    c_center:   The carbon atom where the plane is centered/anchored.
    c_attached: The other carbon atom (defines the main longitudinal axis).
    h_attached: The hydrogen atom defining the dihedral orientation.
    """
    # Vector 1: Along the C-C bond (Longitudinal axis)
    v1 = c_attached - c_center
    length_v1 = np.linalg.norm(v1)
    v1_unit = v1 / length_v1

    # Vector along the C-H bond
    v_ch = h_attached - c_center

    # Normal to the plane
    nv = np.cross(v1_unit, v_ch)
    nv /= np.linalg.norm(nv)

    # Vector 2: Perpendicular to C-C bond, inside the H-C-C plane
    v2_unit = np.cross(nv, v1_unit)

    def xyz(p):
        return {"x": float(p[0]), "y": float(p[1]), "z": float(p[2])}

    # Grid parameters
    n_lines = 9
    radius = 0.015

    # We shift the center of the grid slightly forward along the C-C bond
    # so the plane cleanly spans across the space between both carbons.
    grid_center = c_center + (v1 * 0.5)

    # Longitudinal span (along C-C bond)
    span_v1 = np.linspace(-length_v1 * 0.6, length_v1 * 0.6, n_lines)
    # Lateral span (outwards towards the Hydrogen)
    span_v2 = np.linspace(0, size, n_lines)

    # Draw lines parallel to the C-C bond (v1)
    for t2 in span_v2:
        start = grid_center + t2 * v2_unit + span_v1[0] * v1_unit
        end = grid_center + t2 * v2_unit + span_v1[-1] * v1_unit
        view.addCylinder(
            {
                "start": xyz(start),
                "end": xyz(end),
                "radius": radius,
                "color": color,
                "fromCap": 0,
                "toCap": 0,
            }
        )

    # Draw lines perpendicular to the C-C bond (v2)
    for t1 in span_v1:
        start = grid_center + t1 * v1_unit + span_v2[0] * v2_unit
        end = grid_center + t1 * v1_unit + span_v2[-1] * v2_unit
        view.addCylinder(
            {
                "start": xyz(start),
                "end": xyz(end),
                "radius": radius,
                "color": color,
                "fromCap": 0,
                "toCap": 0,
            }
        )

    # Clean outer border outline
    corners = [
        grid_center + span_v1[-1] * v1_unit + span_v2[-1] * v2_unit,
        grid_center + span_v1[0] * v1_unit + span_v2[-1] * v2_unit,
        grid_center + span_v1[0] * v1_unit + span_v2[0] * v2_unit,
        grid_center + span_v1[-1] * v1_unit + span_v2[0] * v2_unit,
    ]
    for i in range(4):
        view.addCylinder(
            {
                "start": xyz(corners[i]),
                "end": xyz(corners[(i + 1) % 4]),
                "radius": radius * 1.5,
                "color": color,
                "fromCap": 1,
                "toCap": 1,
            }
        )


# ── 3. Color Helper & Visualise ──────────────────────────────────────────────
def blend_hex_colors(hex1, hex2):
    """Blends two hex colors evenly (50/50 split)."""
    # Remove '#' and convert to RGB integers
    rgb1 = [int(hex1.lstrip("#")[i:i+2], 16) for i in (0, 2, 4)]
    rgb2 = [int(hex2.lstrip("#")[i:i+2], 16) for i in (0, 2, 4)]
    # Average them
    blended_rgb = [int((c1 + c2) / 2) for c1, c2 in zip(rgb1, rgb2)]
    # Convert back to hex string
    return f"0x{blended_rgb[0]:02x}{blended_rgb[1]:02x}{blended_rgb[2]:02x}"

# Colors
purple_color = "#7F77DD"
teal_color = "#1D9E75"
carbon_color = blend_hex_colors(purple_color, teal_color)

# Convert plane hex strings to py3Dmol's expected 0xHEX format for atom styles
purple_hex = f"0x{purple_color.lstrip('#')}"
teal_hex = f"0x{teal_color.lstrip('#')}"

# Initialize view
view = py3Dmol.view(width=600, height=440)
view.addModel(Chem.MolToMolBlock(mol), "mol")

# 1. Base style for all standard atoms (The rest of the Hydrogens)
view.setStyle(
    {"elem": "H"},
    {
        "stick": {"radius": 0.2, "color": "white"},
        "sphere": {"scale": 0.28, "color": "white"},
    },
)

# 2. Apply blended color to both Carbon atoms (Indices 0 and 1)
view.setStyle(
    {"elem": "C"},
    {
        "stick": {"radius": 0.1, "color": carbon_color},
        "sphere": {"scale": 0.28, "color": carbon_color},
    },
)

# 3. Apply Purple color to the specific H atom on C0 (Index 2)
view.setStyle(
    {"index": 2},
    {
        "stick": {"radius": 0.1, "color": purple_hex},
        "sphere": {"scale": 0.32, "color": purple_hex}, # Made slightly larger for emphasis
    },
)

# 4. Apply Teal color to the specific H atom on C1 (Index 5)
view.setStyle(
    {"index": 5},
    {
        "stick": {"radius": 0.1, "color": teal_hex},
        "sphere": {"scale": 0.32, "color": teal_hex}, # Made slightly larger for emphasis
    },
)

# Add labels A, B, C, D for the dihedral atoms
conf_mol = mol.GetConformer(0)

# Atom A (H on C0, index 2)
pos_A = conf_mol.GetAtomPosition(2)
view.addLabel("A", {
    "position": {"x": pos_A.x, "y": pos_A.y + 0.5, "z": pos_A.z},
    "fontColor": "black",
    "backgroundColor": "white",
    "fontSize": 12,
    "backgroundOpacity": 0.8
})

# Atom B (C0, index 0)
pos_B = conf_mol.GetAtomPosition(0)
view.addLabel("B", {
    "position": {"x": pos_B.x, "y": pos_B.y + 0.5, "z": pos_B.z},
    "fontColor": "black",
    "backgroundColor": "white",
    "fontSize": 12,
    "backgroundOpacity": 0.8
})

# Atom C (C1, index 1)
pos_C = conf_mol.GetAtomPosition(1)
view.addLabel("C", {
    "position": {"x": pos_C.x, "y": pos_C.y + 0.5, "z": pos_C.z},
    "fontColor": "black",
    "backgroundColor": "white",
    "fontSize": 12,
    "backgroundOpacity": 0.8
})

# Atom D (H on C1, index 5)
pos_D = conf_mol.GetAtomPosition(5)
view.addLabel("D", {
    "position": {"x": pos_D.x, "y": pos_D.y + 0.5, "z": pos_D.z},
    "fontColor": "black",
    "backgroundColor": "white",
    "fontSize": 12,
    "backgroundOpacity": 0.8
})

# ── 4. Draw planes & render ──────────────────────────────────────────────────
# Purple plane: Defined by C0 -> C1 and the first Hydrogen on C0
draw_dihedral_plane(view, c0, c1, h_c0[0], color=purple_color, size=1.3)

# Teal plane: Defined by C1 -> C0 and the first Hydrogen on C1
draw_dihedral_plane(view, c1, c0, h_c1[0], color=teal_color, size=1.3)

view.setBackgroundColor("white")
view.zoomTo()
view.show()

---
# Checkpoint 1

### **Question:**
*   Examine the 3D model below, which shows a molecule in an **eclipsed conformation**. What is the approximate dihedral angle of this conformation (1-2-3-4)? Enter your answer in cell 2.3.

Please type your answer in degrees (°).
---

In [ ]:
#@title Cell 2.2: Checkpoint 1
# Temporarily create an ethane molecule for the checkpoint
ethane_smi = "CC"
ethane_mol = Chem.MolFromSmiles(ethane_smi)
ethane_mol = Chem.AddHs(ethane_mol)

# CPK color scheme (moved from cell 77d2d50c to resolve NameError)
element_colors = {
    "H":  "#FFFFFF", # White
    "C":  "#909090", # Grey
    "N":  "#2944cf", # Blue
    "O":  "#d60d0d", # Red
    "F":  "#37c4a9", # Teal/Green
    "Cl": "#41bf41", # Green
    "P":  "#FF8000", # Orange
    "S":  "#b5b50d", # Yellow
}

# Embed and optimize (optional, but good practice)
AllChem.EmbedMolecule(ethane_mol, AllChem.ETKDGv3())
AllChem.MMFFOptimizeMolecule(ethane_mol)

# Find the central C-C bond (0-1 for ethane)
atom_a_idx = 0 # first carbon
atom_b_idx = 1 # second carbon

# Find an H on the first carbon (atom_a) that is not atom_b
# Find an H on the second carbon (atom_b) that is not atom_a

# Get neighbors of C0 and C1 to find the atoms for the dihedral
c0_neighbors = [a.GetIdx() for a in ethane_mol.GetAtomWithIdx(0).GetNeighbors() if a.GetIdx() != 1]
c1_neighbors = [a.GetIdx() for a in ethane_mol.GetAtomWithIdx(1).GetNeighbors() if a.GetIdx() != 0]

# Ensure we have at least one hydrogen for each end of the dihedral for visualization
if not c0_neighbors or not c1_neighbors:
    raise ValueError("Could not find suitable atoms for dihedral on ethane.")

atom_a_dihedral = c0_neighbors[0] # An atom attached to the front carbon
atom_d_dihedral = c1_neighbors[0] # An atom attached to the back carbon

# Set the dihedral angle to 0 degrees for an eclipsed conformation
rdMolTransforms.SetDihedralDeg(ethane_mol.GetConformer(), atom_a_dihedral, atom_a_idx, atom_b_idx, atom_d_dihedral, 0.0)

# Display the eclipsed molecule
view_eclipsed = py3Dmol.view(width=500, height=350)
view_eclipsed.setBackgroundColor("white")
view_eclipsed.addModel(Chem.MolToMolBlock(ethane_mol), "mol")

for elem, color in element_colors.items():
    view_eclipsed.setStyle({"elem": elem}, {
        "stick": {"color": color, "radius": 0.1},
        "sphere": {"scale": 0.25, "color": color}
    })

# Add labels for the dihedral atoms for clarity
conf_ecl = ethane_mol.GetConformer(0)
diaphragm_atoms = [atom_a_dihedral, atom_a_idx, atom_b_idx, atom_d_dihedral]
labels = ['1', '2', '3', '4'] # Custom labels

for i, atom_idx in enumerate(diaphragm_atoms):
    pos = conf_ecl.GetAtomPosition(atom_idx)
    view_eclipsed.addLabel(labels[i], {
        "position": {"x": pos.x, "y": pos.y + 0.5, "z": pos.z},
        "fontColor": "black",
        "backgroundColor": "white",
        "fontSize": 12,
        "backgroundOpacity": 0.8
    })

view_eclipsed.zoomTo()
view_eclipsed.show()


In [ ]:
#@title Cell 2.3: Run This Cell and Enter Your Answer Here

while True:
  try:
    answer1 = int(input("What is the dihedral angle made up of atom 1-2-3-4? "))
    if answer1 not in [int(chr(48)), int(chr(51) + chr(54) + chr(48))]:
      print("Your answer for checkpoint 1 is incorrect. Please try to answer the question again. Let your professors know if you need help! ")
    else:
      print("Your answer for checkpoint 1 is correct!")
      molecule_library = {
          "ethane":           "CC",
          "propane":          "CCC",
          "pentane":          "CCCCC",
          "isobutane":        "CC(C)C",
          "cyclohexane":      "C1CCCCC1",
          "ethanol":          "CCO",
          "acetic acid":      "CC(=O)O",
          "acetone":          "CC(C)=O",
          "ethylenediamine":  "NCCN",
          "butadiene":        "C=CC=C",
          "bipyridine":       "c2ccc(c1ccccn1)nc2",
          "aspirin":          "CC(=O)OC1=CC=CC=C1C(=O)O",
      }
      break
  except ValueError:
    print("Please enter a number.")

## 2d. Staggered vs. Eclipsed Conformations

For a simple molecule like ethane (C₂H₆), rotating around the C–C bond alternates between
two extremes:

- **Staggered** (dihedral = 60°, 180°, 300°): Hydrogen atoms on adjacent carbons are as far
  apart as possible. These are **energy minima** — the most stable conformations.
- **Eclipsed** (dihedral = 0°, 120°, 240°): Hydrogens are directly in front of each other.
  Torsional strain from overlapping electron clouds raises the energy. These are **energy maxima**.
<img src="https://www.chemistrysteps.com/wp-content/uploads/2024/10/Eclipsed-and-Staggered-Conformations.png" width="400">

Source: https://www.chemistrysteps.com/staggered-and-eclipsed-conformations/

For butane (C₄H₁₀), two staggered conformations are NOT equivalent:
- **Anti** (dihedral ≈ 180°): The two methyl groups are on opposite sides — lowest energy.
- **Gauche** (dihedral ≈ 60°, 300°): Methyl groups are at 60° to each other — slightly higher
  energy due to steric strain. Still a minimum, but less stable than anti.
<img src="https://d3rw207pwvlq3a.cloudfront.net/attachments/000/025/228/original/CHEM123_Final_Theo_img101.png?1539132342" width="400">

Source: https://www.wizeprep.com/online-courses/17257/chapter/13/core/10/4

---

# 3. Section Overview: SMILES Notation

Computational chemistry software needs a way to represent molecules as text. One of the most common formats is **SMILES** (Simplified Molecular Input Line Entry System), which encodes a molecule's atoms and connectivity using a short character string.

For example:

| Molecule | SMILES |
|---|---|
| Ethane | `CC` |
| Ethanol | `CCO` |
| Propane | `CCC` |
| Benzene | `c1ccccc1` |

Throughout this notebook, molecules will often be specified using their SMILES strings. You do **not** need to learn the full SMILES syntax to complete this activity.

## * Generating SMILES Strings

If you know the structure of a molecule but not its SMILES string, you can use an online molecular editor:

**MolView:** https://molview.org/

Instructions:

1. Open MolView in your browser.
2. Draw the molecule using the drawing tools.
3. Hover over tools, in the drop down menu, click information card. A lot of useful information about the molecule you just drew will be displayed.
4. Copy the displayed **SMILES** string. Either Canonical SMILES or Isomeric SMILES will work.
5. Paste the SMILES string into the code cells in this notebook when prompted.

For example, drawing propane produces the SMILES string:

```text
CCC
```

which can then be used directly in RDKit, a library used in this notebook:

```python
smiles = "CCC"
```

### Run the cell below, try to draw Butane (C4H10) in MolView, copy its SMILES string and paste it in the code cell you just ran then hit Enter.


In [ ]:
#@title Cell 3.1: Please input your Butane SMILES string

# Check if 'molecule_library' is defined. If not, instruct the user to complete Checkpoint 1.
if 'molecule_library' not in globals():
    print("Error: 'molecule_library' is not defined. Please go back to Cell 2.3 and ensure Checkpoint 1 is completed successfully.")
else:
  # Prompt user to enter 'butane' and add it if correct
  while True:
      user_input = input("Please enter the SMILES string for 'butane': ")
      if user_input == "CCCC":
          molecule_library["butane"] = "CCCC"
          print("Butane successfully added to the library.")
          break
      else:
          print("Incorrect input. Please try again.")

  print("Molecule library loaded.")
  print(f"Available molecules: {', '.join(molecule_library.keys())}")

---

# 4. Visualizing 2D Structures

Before we build 3D models, let's draw 2D skeletal structures for the molecules in our
library. RDKit can render these directly from SMILES — no internet connection required.

The cell below displays all molecules in the reference library as a grid.
This is a quick sanity check: if a SMILES string is invalid, RDKit will display a blank
panel rather than raising an error.

In [ ]:
#@title Cell 4.1: Show All 2D Structures of Available Molecules
# ── SMILES Reference Library ──────────────────────────────────────────────────
# Each entry: "common name": "SMILES string"
# Students can add their own entries here.

# ── Display 2D structures for all library molecules ───────────────────────────

mols_2d = []
labels_2d = []

for name, smi in molecule_library.items():
    mol = Chem.MolFromSmiles(smi)
    if mol:
        mols_2d.append(mol)
        labels_2d.append(name)
    else:
        print(f"  Warning: invalid SMILES for '{name}': {smi}")

img = Draw.MolsToGridImage(
    mols_2d,
    molsPerRow=5,
    subImgSize=(300, 250),
    legends=labels_2d
)
display(img)

**Checkpoint:** Can you identify which structures match the SMILES strings from Section 3b?
Look for the functional groups you recognized earlier.

---

# 5. Building a 3D Molecular Model

Going from a flat SMILES string to a 3D structure requires two steps:

1. **Embedding** — placing atoms in 3D space using distance geometry (ETKDGv3 algorithm).
2. **Optimization** — minimizing the energy with a force field (MMFF94) so bond lengths,
   angles, and torsions are chemically reasonable.

We then color atoms using the standard **CPK color scheme** (named after Corey, Pauling,
and Koltun), which you may have seen on physical molecular models:

| Element | Color |
|---|---|
| H | White |
| C | Grey |
| N | Blue |
| O | Red |
| S | Yellow |
| P | Orange |
| F | Teal |
| Cl | Green |

Run the cell, then use your mouse to **rotate, zoom, and pan** the 3D model.

In [ ]:
#@title Cell 5.1: Choose your molecule

# Get the keys from the molecule_library dictionary to populate the dropdown
molecule_names = list(molecule_library.keys())

# Create a Label widget for the description
description_label = widgets.Label(value='Choose your molecule:')

# Create a Dropdown widget without a description, as it's now handled by the Label
molecule_dropdown = widgets.Dropdown(
    options=molecule_names,
    value="ethane",
    disabled=False,
    layout={'width': '300px'} # Set a fixed width for the dropdown
)

# Display the label and dropdown using a VBox to place them on separate lines
display(widgets.VBox([description_label, molecule_dropdown]))

# Assign the selected value to molecule_choice
# This will update whenever the dropdown value changes
def on_change_molecule(change):
    global molecule_choice
    molecule_choice = change.new

molecule_dropdown.observe(on_change_molecule, names='value')

# Initialize molecule_choice with the default value of the dropdown
molecule_choice = molecule_dropdown.value

In [ ]:
import builtins
input = builtins.input # Restore the built-in input function if it was overwritten

#@title Cell 5.2: Or if you would like to try other molecules that are not in the predefined library:

smiles_string_input  = input("Enter your SMILES string of the molecule you have chosen: ")
molecule_choice = smiles_string_input

In [ ]:
#@title Cell 5.3: Create a 3D viewer of the molecule
# ── CPK color scheme ──────────────────────────────────────────────────────────
element_colors = {
    "H":  "#FFFFFF", # White
    "C":  "#909090", # Grey
    "N":  "#2944cf", # Blue
    "O":  "#d60d0d", # Red
    "F":  "#37c4a9", # Teal/Green
    "Cl": "#41bf41", # Green
    "P":  "#FF8000", # Orange
    "S":  "#b5b50d", # Yellow
}

# ── Build 3D model ────────────────────────────────────────────────────────────
# ↓ Change this to any molecule name from molecule_library, or enter a SMILES directly.
# molecule_choice = "ethanol"   # ← only line you need to change

smiles_input = molecule_library.get(molecule_choice, molecule_choice)
mol = Chem.MolFromSmiles(smiles_input)

if mol is None:
    raise ValueError(f"Could not parse SMILES: '{smiles_input}'. Check the string for typos.")

mol = Chem.AddHs(mol)           # Add explicit hydrogens for 3D geometry

params = AllChem.ETKDGv3()
params.randomSeed = 42           # Fixed seed → reproducible geometry
AllChem.EmbedMolecule(mol, params)
AllChem.MMFFOptimizeMolecule(mol)

carbon_idxs = [a.GetIdx() for a in mol.GetAtoms() if a.GetSymbol() == 'C']
conf = mol.GetConformer()
# The following loop attempts to set dihedral angles for all possible carbon chains
# but incorrectly assumes consecutive carbons in `carbon_idxs` are bonded, causing a ValueError.
# It's also not necessary for simply displaying the 3D model, as the molecule is already embedded and optimized.
# for i in range(len(carbon_idxs) - 3):
#     rdMolTransforms.SetDihedralDeg(conf, carbon_idxs[i], carbon_idxs[i+1], carbon_idxs[i+2], carbon_idxs[i+3], 180.0)
#     # Get the dihedral angle after setting it, to define 'deg'
#     deg = rdMolTransforms.GetDihedralDeg(conf, carbon_idxs[i], carbon_idxs[i+1], carbon_idxs[i+2], carbon_idxs[i+3])
#     # print(f"Dihedral C{i+1}-C{i+2}-C{i+3}-C{i+4}: {deg:.1f}°")

print(f"Molecule: {molecule_choice}")
print(f"SMILES: {smiles_input}")
print(f"Atoms (with H): {mol.GetNumAtoms()}")
print(f"Bonds: {mol.GetNumBonds()}")

# ── Render in 3D ──────────────────────────────────────────────────────────────

view = py3Dmol.view(width=600, height=400)
view.setBackgroundColor("white")
view.addModel(Chem.MolToMolBlock(mol), "mol")

for elem, color in element_colors.items():
    view.setStyle({"elem": elem}, {
        "stick": {"color": color, "radius": 0.1},
        "sphere": {"scale": 0.25, "color": color}
    })

# Add non-hydrogen atom indexes
conf = mol.GetConformer(0)
for atom in mol.GetAtoms():
    if atom.GetSymbol() != 'H':  # Exclude hydrogen atoms
        idx = atom.GetIdx()
        pos = conf.GetAtomPosition(idx)
        view.addLabel(str(idx + 1), {
            "position": {"x": pos.x, "y": pos.y + 0.5, "z": pos.z},
            "fontColor": "black",
            "backgroundColor": "white",
            "fontSize": 12,
            "backgroundOpacity": 0.8
        })

view.zoomTo()
view.zoom(2)
view.show()

**Explore the model:** Can you identify every atom by its color? Can you count the hydrogens?
How does the 3D structure compare to the 2D skeletal formula?

---

# 6. Conformational Energy Scanner

Now we'll rotate around the central C–C bond of our molecule and record the MMFF94 energy
at every dihedral angle from 0° to 360°. This produces the **rotational energy profile**.

The scanner:
1. Identifies the best central rotatable C–C bond automatically.
2. Sets the dihedral to each angle (every 1°, so 361 points total).
3. Minimizes the rest of the molecule lightly while holding the dihedral atoms fixed.
4. Records the energy in kJ/mol relative to the global minimum.



In [ ]:
#@title Cell 6.1: Energy Calculations
%matplotlib widget

# ── 7a. Find the dihedral atoms ───────────────────────────────────────────────

def get_rotation_atoms(mol):
    """
    Finds a central rotatable C–C single bond and returns four atoms
    (a, b, c, d) defining the dihedral a–b–c–d.
    Returns the middle-most rotatable bond to capture the most
    representative conformation for longer chains.
    """
    rot_bonds = []
    for bond in mol.GetBonds():
        a1 = bond.GetBeginAtom()
        a2 = bond.GetEndAtom()
        if (bond.GetBondTypeAsDouble() == 1.0
                and not bond.IsInRing()
                and a1.GetSymbol() == 'C'
                and a2.GetSymbol() == 'C'):
            rot_bonds.append((a1.GetIdx(), a2.GetIdx()))

    if not rot_bonds:
        raise ValueError("No rotatable C–C bonds found in this molecule. "
                         "Try butane, pentane, or another acyclic alkane.")

    b_idx, e_idx = rot_bonds[len(rot_bonds) // 2]

    def best_neighbor(atom_idx, exclude_idx):
        neighbors = [n for n in mol.GetAtomWithIdx(atom_idx).GetNeighbors()
                     if n.GetIdx() != exclude_idx]
        carbons = [n for n in neighbors if n.GetSymbol() == 'C']
        return (carbons[0] if carbons else neighbors[0]).GetIdx()

    a = best_neighbor(b_idx, e_idx)
    d = best_neighbor(e_idx, b_idx)
    return a, b_idx, e_idx, d

atom_a, atom_b, atom_c, atom_d = get_rotation_atoms(mol)
print(f"Central bond selected: C{atom_b}–C{atom_c}")
print(f"Dihedral atoms: "
      f"{mol.GetAtomWithIdx(atom_a).GetSymbol()}{atom_a}–"
      f"C{atom_b}–C{atom_c}–"
      f"{mol.GetAtomWithIdx(atom_d).GetSymbol()}{atom_d}")

mmff_props = AllChem.MMFFGetMoleculeProperties(mol, mmffVariant="MMFF94")

# ── 7b. Scan all dihedral angles ──────────────────────────────────────────────
# For each angle, we temporarily fix the four dihedral atoms in place, then
# minimize the rest of the molecule. This isolates the torsional energy
# contribution from the bond we're rotating.

print("\nComputing energy profile...")

angles   = np.arange(0, 361, 1)
energies = []

for angle in angles:
    mol_tmp = Chem.Mol(mol)
    rdMolTransforms.SetDihedralDeg(
        mol_tmp.GetConformer(), atom_a, atom_b, atom_c, atom_d, float(angle)
    )
    ff = AllChem.MMFFGetMoleculeForceField(mol_tmp, mmff_props)
    # Position constraints prevent the reference atoms from drifting during minimization
    for atom_idx in [atom_a, atom_b, atom_c, atom_d]:
        ff.MMFFAddPositionConstraint(atom_idx, 0, 1e4)
    ff.Minimize(maxIts=500)
    energies.append(ff.CalcEnergy() * 4.184)   # kcal/mol → kJ/mol

energies          = np.array(energies)
min_absolute_energy = energies.min()
energies          = energies - min_absolute_energy   # Set minimum = 0 kJ/mol

min_energy    = energies.min()
max_energy    = energies.max()
barrier_height = max_energy - min_energy

print(f"\nDone!")
print(f"  Barrier height (ΔE): {barrier_height:.2f} kJ/mol")
print(f"  Most stable conformation: {angles[np.argmin(energies)]}°")
print(f"  Least stable conformation: {angles[np.argmax(energies)]}°")

---
# Checkpoint 2

### **Question:**
*   For Ethane, which conformer is the most stable?

Please enter one of these as your answer: Eclipsed, Staggered, Gauche, Anti.
---

In [ ]:
#@title Cell 6.2: Checkpoint 2

while True:
    answer2 = input("For Ethane, which conformer is the most stable? (Eclipsed, Staggered, Gauche, Anti) ").strip().lower()
    if answer2 == chr(115) + chr(116) + chr(97) + chr(103) + chr(103) + chr(101) + chr(114) + chr(101) + chr(100):
        print("Your answer for checkpoint 2 is correct! You can now proceed to the next cell.")
        # Initialize the rotation atoms if the answer is correct
        default_a, default_b, default_c, default_d = get_rotation_atoms(mol)
        break
    elif answer2 == chr(101) + chr(99) + chr(108) + chr(105) + chr(112) + chr(115) + chr(101) + chr(100):
        print("Your answer for checkpoint 2 is incorrect. Please try again.")
    elif answer2 == chr(103) + chr(97) + chr(117) + chr(99) + chr(104) + chr(101) or answer2 == chr(97) + chr(110) + chr(116) + chr(105):
        print("In ethane, only eclipsed and staggered conformations exist; gauche and anti conformations are observed in larger alkanes such as butane. Please try again")
    else:
      print("Invalid input. Please only enter one of the option above.")


---

# 7. Interactive Visualization

The widget below combines the 3D molecular viewer and the energy profile plot.
Drag the **Dihedral Angle** slider to rotate around the central C–C bond and watch:
- The 3D model update in real time
- The orange dot trace the energy curve

**What to look for:**
- Where are the minima? Are they all the same depth (symmetric molecule) or different (asymmetric)?
- Where are the maxima? What does the 3D structure look like at those points?
- How large is the rotational barrier compared to thermal energy at room temperature
  (~2.5 kJ/mol)? Can the molecule rotate freely?

## In this code cell below, you can specify what bond you would like to rotate in the molecule. For example, Butane rotating bond is set to the middle bond by default. In the code cell below you can change the static atom to 1 and spinning atom to 2 to have the bond on the side rotate.

In [ ]:
#@title Cell 7.1: Conformational Energy Scanner

# ── 1. Molecule setup ─────────────────────────────────────────────────────────

def get_rotation_atoms(mol, b_idx_user=None, c_idx_user=None):
    if b_idx_user is not None and c_idx_user is not None:
        atom_b = b_idx_user
        atom_c = c_idx_user
        # Verify if they form a bond and are carbons
        bond = mol.GetBondBetweenAtoms(atom_b, atom_c);
        if bond is None or mol.GetAtomWithIdx(atom_b).GetSymbol() != 'C' or mol.GetAtomWithIdx(atom_c).GetSymbol() != 'C':
            raise ValueError(f"Atoms {atom_b+1} and {atom_c+1} do not form a C-C bond or are not both carbons.")
        if bond.IsInRing():
            raise ValueError(f"The bond between C{atom_b+1} and C{atom_c+1} is in a ring. Rotating it will distort the molecule.")
    else:
        rot_bonds = []
        for bond in mol.GetBonds():
            a1 = bond.GetBeginAtom()
            a2 = bond.GetEndAtom()
            if (bond.GetBondTypeAsDouble() == 1.0
                    and not bond.IsInRing()
                    and a1.GetSymbol() == 'C'
                    and a2.GetSymbol() == 'C'):
                rot_bonds.append((a1.GetIdx(), a2.GetIdx()))

        if not rot_bonds:
            raise ValueError("No rotatable C-C bonds found.")
        atom_b, atom_c = rot_bonds[len(rot_bonds) // 2]

    def best_neighbor(atom_idx, exclude_idx):
        neighbors = [n for n in mol.GetAtomWithIdx(atom_idx).GetNeighbors()
                     if n.GetIdx() != exclude_idx]
        carbons = [n for n in neighbors if n.GetSymbol() == 'C']
        if carbons:
            return carbons[0].GetIdx()
        elif neighbors:
            return neighbors[0].GetIdx()
        else:
            return atom_idx

    atom_a = best_neighbor(atom_b, atom_c)
    atom_d = best_neighbor(atom_c, atom_b)
    return atom_a, atom_b, atom_c, atom_d


def run_conformational_scan(b_idx_user=None, c_idx_user=None):
    # Notice: No global variables. Everything is safely enclosed per-run.
    initial_angle = 0

    try:
        adjusted_b_idx = b_idx_user - 1 if b_idx_user is not None else None
        adjusted_c_idx = c_idx_user - 1 if c_idx_user is not None else None
        atom_a, atom_b, atom_c, atom_d = get_rotation_atoms(mol, adjusted_b_idx, adjusted_c_idx)
    except ValueError as e:
        print(f"Error selecting bond: {e}")
        return widgets.Label(value=f"Error: {e}")

    print(f"Rotating bond: C{atom_b}–C{atom_c}")
    print(f"Dihedral: {mol.GetAtomWithIdx(atom_a).GetSymbol()}{atom_a} – "
          f"C{atom_b} – C{atom_c} – "
          f"{mol.GetAtomWithIdx(atom_d).GetSymbol()}{atom_d}")

    mmff_props = AllChem.MMFFGetMoleculeProperties(mol, mmffVariant="MMFF94")

    mol_initial_display = Chem.Mol(mol)
    rdMolTransforms.SetDihedralDeg(mol_initial_display.GetConformer(), atom_a, atom_b, atom_c, atom_d, float(initial_angle))
    ff_initial = AllChem.MMFFGetMoleculeForceField(mol_initial_display, mmff_props)
    ff_initial.MMFFAddPositionConstraint(atom_a, 0, 1e4)
    ff_initial.MMFFAddPositionConstraint(atom_b, 0, 1e4)
    ff_initial.MMFFAddPositionConstraint(atom_c, 0, 1e4)
    ff_initial.MMFFAddPositionConstraint(atom_d, 0, 1e4)
    ff_initial.Minimize(maxIts=500)

    view = py3Dmol.view(width=700, height=420)
    view.setBackgroundColor("white")
    view.removeAllModels()
    view.addModel(Chem.MolToMolBlock(mol_initial_display), "mol")
    for e, c in element_colors.items():
        view.setStyle({"elem": e}, {"stick": {"color": c, "radius": 0.1}, "sphere": {"scale": 0.25, "color": c}})

    rotating_bond_color = "#FFFF00"

    view.setStyle({"index": atom_b}, {
        "stick": {"color": rotating_bond_color, "radius": 0.1},
        "sphere": {"scale": 0.25, "color": rotating_bond_color}
    })
    view.setStyle({"index": atom_c}, {
        "stick": {"color": rotating_bond_color, "radius": 0.1},
        "sphere": {"scale": 0.25, "color": rotating_bond_color}
    })

    view.removeAllLabels()
    conf_initial = mol_initial_display.GetConformer(0)
    for atom in mol_initial_display.GetAtoms():
        if atom.GetSymbol() != 'H': # Exclude hydrogen atoms
            idx = atom.GetIdx()
            pos = conf_initial.GetAtomPosition(idx)
            view.addLabel(str(idx + 1), {
                "position": {"x": pos.x, "y": pos.y + 0.5, "z": pos.z},
                "fontColor": "black",
                "backgroundColor": "white",
                "fontSize": 12,
                "backgroundOpacity": 0.8
            })

    # view.update()

    # CRITICAL: Re-initialize properties to ensure clean state for energy loop, matching the old code
    mmff_props = AllChem.MMFFGetMoleculeProperties(mol, mmffVariant="MMFF94")

    # ── 2. Pre-compute energies with position constraints ─────────────────────────
    print("Computing energies...")
    angles = np.arange(0, 361, 1)
    energies_list = []

    for angle in angles:
        mol_tmp = Chem.Mol(mol)
        rdMolTransforms.SetDihedralDeg(mol_tmp.GetConformer(), atom_a, atom_b, atom_c, atom_d, float(angle))
        ff = AllChem.MMFFGetMoleculeForceField(mol_tmp, mmff_props)
        ff.MMFFAddPositionConstraint(atom_a, 0, 1e4)
        ff.MMFFAddPositionConstraint(atom_b, 0, 1e4)
        ff.MMFFAddPositionConstraint(atom_c, 0, 1e4)
        ff.MMFFAddPositionConstraint(atom_d, 0, 1e4)
        ff.Minimize(maxIts=500)
        energies_list.append(ff.CalcEnergy() * 4.184) # Convert to kJ/mol

    energies = np.array(energies_list)
    min_absolute_energy = energies.min()
    energies = energies - min_absolute_energy

    min_energy = energies.min()
    max_energy = energies.max()
    barrier_height = max_energy - min_energy
    print("Done!")

    # ── 3. Layout ─────────────────────────────────────────────────────────────────
    mol_out = widgets.Output(layout=widgets.Layout(width='50%'))
    with mol_out:
        display(view)

    fig, ax = plt.subplots(figsize=(5.5, 4))
    fig.patch.set_facecolor('#f8f8f8')
    ax.set_facecolor('#f8f8f8')
    ax.plot(angles, energies, color='#87cefa', linewidth=2)
    ax.set_xlabel('Dihedral Angle (°)', color='#333333', fontsize=10)
    ax.set_ylabel('Relative Energy (kJ/mol)', color='#333333', fontsize=10) # Changed to kJ/mol
    ax.tick_params(colors='#333333')
    ax.set_xlim(-40, 450)
    ax.set_xticks([0, 60, 120, 180, 240, 300, 360])
    ax.set_ylim(min_energy - 1, max_energy + 1)
    for spine in ax.spines.values():
        spine.set_edgecolor('#cccccc')

    dot, = ax.plot([angles[initial_angle]], [energies[initial_angle]], 'o', color='#FF5722', markersize=9, zorder=5)
    vline = ax.axvline(x=angles[initial_angle], color='#FF5722', linewidth=0.8, linestyle='--', alpha=0.6)
    title = ax.set_title(f'Conformational Energy Profile   |   {energies[initial_angle]:.4f} kJ/mol',
                         color='#333333', fontsize=11)
    plt.tight_layout()

    ax.annotate('', xy=(380, max_energy), xytext=(380, min_energy),
                arrowprops=dict(arrowstyle='->', color='purple', lw=1.2))
    ax.plot([400, 300], [min_energy, min_energy], color='purple', linestyle='--', linewidth=1.5, zorder=4)
    ax.plot([400, 360], [max_energy, max_energy], color='purple', linestyle='--', linewidth=1.5, zorder=4)
    ax.text(388, (min_energy + max_energy) / 2, f'ΔE: {barrier_height:.2f}\nkJ/mol', # Changed to kJ/mol
            color='purple', fontsize=9, ha='left', va='center',
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', boxstyle='round,pad=0.2'))

    plot_out = widgets.Output(layout=widgets.Layout(width='50%'))
    with plot_out:
        plt.show()

    # ── 4. Slider ─────────────────────────────────────────────────────────────────
    slider = widgets.IntSlider(initial_angle, 0, 360, 1, description="Dihedral",
                               continuous_update=True,
                               layout=widgets.Layout(width='98%'))

    def update_slider_and_view(change):
        angle = change['new']

        mol_tmp = Chem.Mol(mol)
        rdMolTransforms.SetDihedralDeg(mol_tmp.GetConformer(), atom_a, atom_b, atom_c, atom_d, float(angle))
        ff = AllChem.MMFFGetMoleculeForceField(mol_tmp, mmff_props)
        ff.MMFFAddPositionConstraint(atom_a, 0, 1e4)
        ff.MMFFAddPositionConstraint(atom_b, 0, 1e4)
        ff.MMFFAddPositionConstraint(atom_c, 0, 1e4)
        ff.MMFFAddPositionConstraint(atom_d, 0, 1e4)
        ff.Minimize(maxIts=500)
        energy_rel = ff.CalcEnergy() * 4.184 - min_absolute_energy # Convert to kJ/mol

        view.removeAllModels()
        view.addModel(Chem.MolToMolBlock(mol_tmp), "mol")
        for e, c in element_colors.items():
            view.setStyle({"elem": e}, {"stick": {"color": c, "radius": 0.1}, "sphere": {"scale": 0.25, "color": c}})

        view.setStyle({"index": atom_b}, {
            "stick": {"color": rotating_bond_color, "radius": 0.1},
            "sphere": {"scale": 0.25, "color": rotating_bond_color}
        })
        view.setStyle({"index": atom_c}, {
            "stick": {"color": rotating_bond_color, "radius": 0.1},
            "sphere": {"scale": 0.25, "color": rotating_bond_color}
        })

        view.removeAllLabels()
        conf_tmp = mol_tmp.GetConformer(0)
        for atom in mol_tmp.GetAtoms():
            if atom.GetSymbol() != 'H': # Exclude hydrogen atoms
                idx = atom.GetIdx()
                pos = conf_tmp.GetAtomPosition(idx)
                view.addLabel(str(idx + 1), {
                    "position": {"x": pos.x, "y": pos.y + 0.5, "z": pos.z},
                    "fontColor": "black",
                    "backgroundColor": "white",
                    "fontSize": 12,
                    "backgroundOpacity": 0.8
                })

        view.update()

        dot.set_data([angle], [energies[angle]])
        vline.set_xdata([angle])
        title.set_text(f'Conformational Energy Profile   |   {energy_rel:.4f} kJ/mol') # Changed to kJ/mol
        fig.canvas.draw_idle()

    slider.observe(update_slider_and_view, names='value')

    return widgets.VBox([slider, widgets.HBox([mol_out, plot_out])])


# --- User input for bond selection ---
# default_a, default_b, default_c, default_d = get_rotation_atoms(mol)
try:
  bond_b_input = widgets.IntText(
      value=default_b + 1,
      description='Static Atom:',
      disabled=False
  )
  bond_c_input = widgets.IntText(
      value=default_c + 1,
      description='Spinning Atom:',
      disabled=False
  )
  set_bond_button = widgets.Button(
      description='Set Rotating Bond',
      disabled=False,
      button_style='',
      tooltip='Click to set the rotating bond',
      icon='check'
  )


  conformational_scanner_output = widgets.Output()

  def on_set_bond_button_clicked(b):
      with conformational_scanner_output:
          conformational_scanner_output.clear_output(wait=True)
          display(run_conformational_scan(b_idx_user=bond_b_input.value, c_idx_user=bond_c_input.value))

  set_bond_button.on_click(on_set_bond_button_clicked)

  # Initial display
  with conformational_scanner_output:
      display(run_conformational_scan())


  display(widgets.HBox([bond_b_input, bond_c_input, set_bond_button]), conformational_scanner_output)

except Exception:
  print("You skipped Checkpoint 2. Please go back to cell 6.2 and make sure that Checkpoint 2 is completed successfully.")



---

# 9. Results & Interpretation

Use your energy profile and the slider to answer the following questions in the
markdown cell (or on paper). Then compare with the discussion below.

## Questions for Reflection

###**Question 1:**

*   At approximately what dihedral angles do you observe energy minima?
What type of conformation (staggered, anti, gauche) does each minimum represent?



###**Question 2:**
*   At approximately what dihedral angles do you observe energy maxima?
What type of conformation (eclipsed, fully eclipsed) does each maximum represent?

###**Question 3:**
*   Is the energy profile symmetric? Why or why not?
*(Hint: think about the symmetry of the molecule you chose.)*

###**Question 4:**
*   Is the rotational barrier small or large compared to room-temperature thermal
energy (~2.5 kJ/mol)? What does this imply about how freely the molecule rotates at 25°C?

###**Question 4:**
*   If you chose butane: what is the energy difference between the anti and gauche
conformations? This is called the **gauche penalty**. Look up a literature value —
how does your computed value compare?

###**Question 5:**
*   For butane, try lining up C2 and C3 atoms and draw Newman projection.


---



# 10. Assessment Questions

Answer each question based on what you observed in this notebook and your course notes.

---

###**Question 1 — SMILES Interpretation**

*   Write the SMILES string for **2-methylbutane** (isopentane) without looking it up.
Check your answer by adding it to `molecule_library` in Section 3c and running
the 2D display cell. Does your 2D structure match what you expected?

*(Hint: the parent chain is butane `CCCC`; a methyl branch on C2 is added with parentheses.)*

---

###**Question 2 — Identifying Conformational Features**

*   Run the conformational scanner on **butane** (set `molecule_choice = "butane"` in Section 6).

*   (a) How many energy minima appear between 0° and 360°?
*   (b) How many energy maxima appear?
*   (c) Identify the global minimum: is it the anti or gauche conformation?

---

###**Question 3 — Comparing Molecules**

*   Run the scanner on **ethane** (`molecule_choice = "ethane"`) and then on **butane**.

*   (a) Compare the barrier heights. Which is larger? Why?
*   (b) How many minima does ethane have between 0° and 360°? How many does butane have?
*   (c) Are all minima in ethane equal in energy? Are all minima in butane equal in energy?
    Explain the difference in terms of molecular structure.
*   (d) Sketch (on paper) what a Newman projection of each minimum looks like.

---

###**Question 4 — Prediction Before Running**

*   Before running the scanner, predict the shape of the energy profile for **propane**
(`molecule_choice = "propane"`):
*   (a) How many minima do you expect? Are they equivalent?
*   (b) Estimate the barrier height: will it be closer to ethane or butane? Explain.
*   (c) Run it and compare to your prediction. Were you correct?

---

---

# 11. Wrap-Up

In this notebook you:
- Learned to **read and write SMILES** notation for organic molecules.
- Get comfortable with running multiple code cells in the correct order.
- Connected computational results to the physical chemistry concepts of **torsional strain,
  steric strain, and conformational stability**.

The same workflow scales to larger, more complex molecules — try aspirin or capsaicin from
the library, or input any molecule you find on PubChem!

---

## Challenge Extensions

1. **Write your own SMILES:** Find a molecule on PubChem, copy its SMILES, paste it into
   `molecule_library`, and run the full analysis. Does the energy profile make chemical sense?

2. **Isobutane vs. butane:** Run both and compare. Isobutane has a tertiary carbon at the
   center. Does that change the barrier or the number of minima?

3. **Heteroatom effect:** Replace one carbon in butane with nitrogen (SMILES: `CCNC`).
   How does the nitrogen change the energy profile? *(Hint: lone pair interactions.)*

4. **Energy units:** The code converts kcal/mol → kJ/mol (multiply by 4.184). Find the line
   in Section 7b where this happens and change it to keep units in kcal/mol. How does the
   plot change? What is the approximate barrier in kcal/mol for butane?

---

## Notebook Sign-Off Checklist

* [ ] **Purpose is clear** from the header without opening code cells.
* [ ] **Every significant decision** has a markdown explanation.
* [ ] **Learning goals** are listed explicitly at the start.
* [ ] **Assessment questions** are answered and submitted.
* [ ] **Variable names** indicate both content and workflow stage.
* [ ] **Kernel restarted** and all cells run top-to-bottom without error.
* [ ] **molecule_choice** was tested with at least two different molecules.